In [1]:
# Parameters
Month = 3
Year = 2569


In [2]:
import time
import pandas as pd
import numpy as np
import os
from pathlib import Path
current_dir = Path.cwd()
# Month =2
# Year = 2569

thai_months = ["", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
mytext = f"{thai_months[Month]} {str(Year)[-2:]}"
excel_file_path1 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด บ_{mytext}_.xlsx"
excel_file_path2 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด บ1_{mytext}_.xlsx"
excel_file_path3 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด น_{mytext}_.xlsx"
excel_file_path4 = current_dir.parent / "01-database" / "ราคากลาง" / f"ไฟล์ราคากลาง ชุด อ_{mytext}_.xlsx"

excel_file_path4

WindowsPath('c:/Users/User/Documents/GitHub/AutomationCPI/01-database/ราคากลาง/ไฟล์ราคากลาง ชุด อ_มี.ค. 69_.xlsx')

## ชุด อ

In [3]:
import warnings
warnings.filterwarnings('ignore', message='All-NaN axis encountered')



try:
    # 1. อ่านข้อมูลจากทุกชีท
    all_sheets = pd.read_excel(excel_file_path4, skiprows=1,engine='calamine', sheet_name=None)

    all_data = []

    for sheet_name, df in all_sheets.items():
        # ตรวจสอบเบื้องต้น: ถ้าชีทมีคอลัมน์น้อยเกินไปให้ข้าม
        if df.shape[1] < 5:
            continue
            
        # คัดลอกข้อมูลและลบแถวที่ 'รหัสรายการ' และ 'ชื่อรายการ' ว่างพร้อมกัน
        # (อ้างอิงตำแหน่ง 0 และ 1 ของชีทต้นฉบับ)
        df_clean = df.dropna(subset=df.columns[[0, 1]], how='all').copy()
        
        if df_clean.empty:
            continue

        # --- ส่วนที่ 1: เตรียมข้อมูลพื้นฐาน ---
        # เก็บข้อมูล 4 คอลัมน์แรกไว้ (Code, ชื่อ, Link, ร้านค้า)
        res = pd.DataFrame()
        res['ชื่อชีท'] = [sheet_name] * len(df_clean)
        res['COMMODITY_CODE'] = df_clean.iloc[:, 0].astype(str).replace('nan', '')
        res['COMMODITY_CODE'] = res['COMMODITY_CODE'].str.zfill(16)
        res['ชื่อรายการ'] = df_clean.iloc[:, 1]
        res['link'] = df_clean.iloc[:, 2]
        res['ประเภทร้านค้า'] = df_clean.iloc[:, 3]

        # --- ส่วนที่ 2: จัดการราคา 4 สัปดาห์ ---
        # คอลัมน์ราคาปกติ/โปร/ภาวะ จะเริ่มที่ index 4, 5, 6 และวนไปทุกๆ 3 คอลัมน์
        for i in range(1, 5): # Week 1 to 4
            base_idx = 4 + (i-1)*3 # W1=4, W2=7, W3=10, W4=13
            
            if base_idx + 1 < df_clean.shape[1]:
                # ดึงราคาปกติ (nw) และ โปร (PROw)
                nw_col = pd.to_numeric(df_clean.iloc[:, base_idx], errors='coerce')
                PROw_col = pd.to_numeric(df_clean.iloc[:, base_idx + 1], errors='coerce')
                
                res[f'NMw{i}'] = nw_col
                res[f'PROw{i}'] = PROw_col
                # คำนวณ week (Min ของ nw และ PROw)
                res[f'week{i}'] = np.nanmin([nw_col, PROw_col], axis=0)
                
                # เก็บค่า 'ภาวะ' (ถ้ามี)
                if base_idx + 2 < df_clean.shape[1]:
                    res[f'ภาวะ{i}'] = df_clean.iloc[:, base_idx + 2]
                else:
                    res[f'ภาวะ{i}'] = np.nan
            else:
                # กรณีคอลัมน์ในชีทมีไม่ถึงสัปดาห์นั้นๆ ให้ใส่ค่าว่าง
                res[f'NMw{i}'], res[f'PROw{i}'], res[f'week{i}'], res[f'ภาวะ{i}'] = np.nan, np.nan, np.nan, np.nan

        # --- ส่วนที่ 3: ค่าเฉลี่ย ---
        # ปกติค่าเฉลี่ยจะอยู่ที่คอลัมน์สุดท้ายของตารางเดิม
        res['ค่าเฉลี่ย'] = pd.to_numeric(df_clean.iloc[:, -1], errors='coerce')

        all_data.append(res)

    # 2. รวมทุกชีทเข้าด้วยกัน
    if all_data:
        WeekAllSheet = pd.concat(all_data, ignore_index=True)
        
        # จัดการรหัสรายการ (COMMODITY_CODE) ไม่ให้ติด .0 หรือเป็นเลขยกกำลัง
        WeekAllSheet['COMMODITY_CODE'] = WeekAllSheet['COMMODITY_CODE'].str.replace(r'\.0$', '', regex=True)
        
        # จัดลำดับคอลัมน์ให้ตรงตามที่ต้องการ
        final_cols = [
            'ชื่อชีท', 'COMMODITY_CODE', 'ชื่อรายการ', 'link', 'ประเภทร้านค้า',
            'NMw1', 'PROw1', 'week1', 'ภาวะ1',
            'NMw2', 'PROw2', 'week2', 'ภาวะ2',
            'NMw3', 'PROw3', 'week3', 'ภาวะ3',
            'NMw4', 'PROw4', 'week4', 'ภาวะ4',
            'ค่าเฉลี่ย'
        ]
        WeekAllSheet = WeekAllSheet[final_cols]
        
        print(f"✅ รวมข้อมูลสำเร็จ: {len(WeekAllSheet)} แถว")
    else:
        print("❌ ไม่พบข้อมูล")

    # WeekAllSheet = WeekAllSheet[['ชื่อชีท','COMMODITY_CODE','ชื่อรายการ','ร้านค้า',
    #                       'week1', 'ภาวะ1', 'week2', 'ภาวะ2','week3', 'ภาวะ3','week4', 'ภาวะ4',]]
    WeekAllSheet['Online'] = 'Online'
    WeekAllSheet = WeekAllSheet[['ชื่อชีท','Online','COMMODITY_CODE','ชื่อรายการ','ประเภทร้านค้า',
                        'week1','week2', 'week3', 'week4', 'ภาวะ1',  'ภาวะ2','ภาวะ3','ภาวะ4']]

    masterF  = current_dir.parent / "01-database" / "Master_CPI" / "Real_Master_CPI.xlsx"
    df_map = pd.read_excel(masterF,sheet_name='shop_manual_cleansing')

    def classify_advanced(shop_name):
        if not isinstance(shop_name, str):
            return "ร้านอื่นๆ"
        
        # วนลูปเช็คตามลำดับแถวใน Excel (สำคัญมาก: บนลงล่าง)
        for _, row in df_map.iterrows():
            target = row['Result']
            
            # เตรียม Primary Keywords (แยก comma)
            primaries = [k.strip() for k in str(row['Primary Keywords']).split(',') if k.strip()]
            
            # เตรียม Sub-Keywords (ถ้ามี)
            subs = [k.strip() for k in str(row['Sub Keywords']).split(',') if k.strip() != 'nan']

            # เช็ค Primary: ต้องเจออย่างน้อย 1 คำ
            if any(p in shop_name for p in primaries):
                # ถ้ามี Sub-Keywords: ต้องเจออย่างน้อย 1 คำในนี้ด้วย
                if subs:
                    if any(s in shop_name for s in subs):
                        return target
                    else:
                        continue # ถ้าไม่เจอ Sub ให้ไปเช็คกฎข้อถัดไป
                
                # ถ้าไม่มี Sub-Keywords แสดงว่าเจอ Primary ก็จบเลย
                return target
                
        return "ร้านอื่นๆ"


    WeekAllSheet['ประเภทร้านค้า'] = WeekAllSheet['ประเภทร้านค้า'].apply(classify_advanced)
    WeekAllSheet


    WeekAllSheet.to_excel(current_dir.parent / "01-database" / "ราคากลาง" / f"Concat_ราคากลาง_{mytext}_ชุดรายสัปดาห์.xlsx", index=False)
    print('มีข้อมูล')
except:
    columns = ['ชื่อชีท','Online','COMMODITY_CODE','ชื่อรายการ','ประเภทร้านค้า','week1','week2', 'week3', 'week4', 'ภาวะ1',  'ภาวะ2','ภาวะ3','ภาวะ4',]
    WeekAllSheet = pd.DataFrame(columns=columns)
    WeekAllSheet.to_excel(current_dir.parent / "01-database" / "ราคากลาง" / f"Concat_ราคากลาง_{mytext}_ชุดรายสัปดาห์.xlsx", index=False)
    print('Columns เปล่า') 

✅ รวมข้อมูลสำเร็จ: 147 แถว


มีข้อมูล


## ชุด บ บ1 น

In [4]:
def center(excel_file_path):
    try:
        # 1. อ่านไฟล์แบบไม่เอา Header (header=None) เพื่อจัดการหัวตารางเอง
        print(f"กำลังอ่านข้อมูลจากไฟล์ '{excel_file_path}'...")
        all_sheets_dict = pd.read_excel(excel_file_path, sheet_name=None, header=None)
        print(f"อ่านไฟล์เสร็จสิ้น พบทั้งหมด {len(all_sheets_dict)} ชีท")

    except Exception as e:
        print(f"เกิดข้อผิดพลาดในการเปิดไฟล์: {e}")
        all_sheets_dict = {}

    def center_logic(df_raw):
        # ลบแถวและคอลัมน์ที่เป็นค่าว่างทั้งหมดทิ้งก่อน เพื่อเช็คเนื้อหาจริง
        df_raw = df_raw.dropna(how='all').dropna(axis=1, how='all')
        
        # --- ตรรกะข้ามชีทเปล่า ---
        # ถ้าชีทไม่มีข้อมูล หรือมีจำนวนแถวน้อยเกินกว่าจะเป็นตารางราคา (ต้องมี Header 2 แถว + ข้อมูล)
        if df_raw.empty or len(df_raw) < 3:
            return pd.DataFrame()

        # --- ตรรกะเดิมของคุณ ---
        try:
            row_store = df_raw.iloc[0].copy()
            row_sub_header = df_raw.iloc[1].copy()
            row_store = row_store.ffill()
            
            df_data = df_raw.iloc[2:].reset_index(drop=True)
            col_code_idx = 0
            col_name_idx = 1
            
            dfs_list = []
            unique_stores = row_store[2:].dropna().unique()
            
            for store_name in unique_stores:
                if "ธ.ค." in str(store_name): continue
                
                store_indices = row_store[row_store == store_name].index
                idx_normal, idx_promo, idx_cond = None, None, None
                
                for idx in store_indices:
                    col_name = str(row_sub_header[idx])
                    if "ราคาปกติ" in col_name: idx_normal = idx
                    elif "ราคาโปร" in col_name: idx_promo = idx
                    elif "ภาวะ" in col_name: idx_cond = idx
                
                if idx_normal is not None:
                    temp_df = pd.DataFrame()
                    temp_df['รหัสรายการ\nCOMMODITY_CODE\n'] = df_data[col_code_idx]
                    temp_df['ชื่อรายการ'] = df_data[col_name_idx]
                    temp_df['ประเภทร้านค้า'] = store_name
                    temp_df['ราคาปกติ'] = df_data[idx_normal]
                    temp_df['ราคาโปร'] = df_data[idx_promo] if idx_promo is not None else None
                    temp_df['ภาวะ'] = df_data[idx_cond] if idx_cond is not None else None
                    dfs_list.append(temp_df)
            
            if not dfs_list: return pd.DataFrame()

            df_final = pd.concat(dfs_list, ignore_index=True)
            df_final = df_final.dropna(subset=['ราคาปกติ', 'ราคาโปร', 'ภาวะ'], how='all')
            return df_final
        except Exception:
            # กรณีโครงสร้างชีทผิดเพี้ยนจนรัน logic ไม่ได้ ให้คืนค่า DF เปล่าเพื่อข้ามชีทนี้
            return pd.DataFrame()

    # Loop รวมข้อมูลจากทุก Sheet
    list_of_dfs = []
    for name, df in all_sheets_dict.items():
        # ตรวจสอบเบื้องต้น ถ้าชีทว่าง (ไม่มีค่าใดๆ เลย) ให้ข้ามทันทีไม่ต้องเข้าฟังก์ชัน logic
        if df.empty or df.dropna(how='all').empty:
            # print(f"⏩ ข้ามชีทเปล่า: {name}")
            continue
            
        dcen = center_logic(df)
        if not dcen.empty:
            list_of_dfs.append(dcen)

    if list_of_dfs:
        allcenter = pd.concat(list_of_dfs, ignore_index=True)
        print(f"✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด {len(allcenter)} แถว")
    else:
        allcenter = pd.DataFrame()
        print("⚠️ ไม่พบข้อมูลที่ใช้งานได้ในไฟล์นี้")
        
    return allcenter

In [5]:
cen1 = center(excel_file_path1)
cen2 = center(excel_file_path2)
cen3 = center(excel_file_path3)

all_centers = pd.concat([cen1, cen2, cen3], ignore_index=True)

กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด บ_มี.ค. 69_.xlsx'...


อ่านไฟล์เสร็จสิ้น พบทั้งหมด 66 ชีท


✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 1341 แถว
กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด บ1_มี.ค. 69_.xlsx'...


อ่านไฟล์เสร็จสิ้น พบทั้งหมด 18 ชีท
✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 407 แถว
กำลังอ่านข้อมูลจากไฟล์ 'c:\Users\User\Documents\GitHub\AutomationCPI\01-database\ราคากลาง\ไฟล์ราคากลาง ชุด น_มี.ค. 69_.xlsx'...


อ่านไฟล์เสร็จสิ้น พบทั้งหมด 23 ชีท
✅ ประมวลผลเสร็จสิ้น ได้ข้อมูลทั้งหมด 46 แถว


In [6]:
try:
    masterF  = current_dir.parent / "01-database" / "Master_CPI" / "Real_Master_CPI.xlsx"
    df_map = pd.read_excel(masterF,sheet_name='shop_manual_cleansing')

    def classify_advanced(shop_name):
        if not isinstance(shop_name, str):
            return "ร้านอื่นๆ"
        
        # วนลูปเช็คตามลำดับแถวใน Excel (สำคัญมาก: บนลงล่าง)
        for _, row in df_map.iterrows():
            target = row['Result']
            
            # เตรียม Primary Keywords (แยก comma)
            primaries = [k.strip() for k in str(row['Primary Keywords']).split(',') if k.strip()]
            
            # เตรียม Sub-Keywords (ถ้ามี)
            subs = [k.strip() for k in str(row['Sub Keywords']).split(',') if k.strip() != 'nan']

            # เช็ค Primary: ต้องเจออย่างน้อย 1 คำ
            if any(p in shop_name for p in primaries):
                # ถ้ามี Sub-Keywords: ต้องเจออย่างน้อย 1 คำในนี้ด้วย
                if subs:
                    if any(s in shop_name for s in subs):
                        return target
                    else:
                        continue # ถ้าไม่เจอ Sub ให้ไปเช็คกฎข้อถัดไป
                
                # ถ้าไม่มี Sub-Keywords แสดงว่าเจอ Primary ก็จบเลย
                return target
                
        return "ร้านอื่นๆ"

    # 3. นำไปใช้งาน
    allcenter_clean = all_centers.copy()
    # เปลี่ยนเป็นชื่อคอลัมน์ที่ถูกต้อง
    allcenter_clean['ประเภทร้านค้า'] = allcenter_clean['ประเภทร้านค้า'].apply(classify_advanced)

    allcenter_clean = allcenter_clean.rename(columns={'รหัสรายการ\nCOMMODITY_CODE\n':'COMMODITY_CODE','ชื่อรายการ':'DESCRIPTION'})
    allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION', 'ประเภทร้านค้า', 'ราคาปกติ','ราคาโปร','ภาวะ']]
    allcenter_clean['COMMODITY_CODE'] = allcenter_clean['COMMODITY_CODE'].astype(str).str.zfill(16)
    type(allcenter_clean['COMMODITY_CODE'][1])


    # Step 1: Convert price columns to numeric, coercing errors to NaN
    allcenter_clean['ราคาปกติ'] = pd.to_numeric(
        allcenter_clean['ราคาปกติ'], 
        errors='coerce'
    )
    allcenter_clean['ราคาโปร'] = pd.to_numeric(
        allcenter_clean['ราคาโปร'], 
        errors='coerce'
    )
    # Element-wise minimum for each row (axis=1)
    allcenter_clean['ราคากลาง'] = allcenter_clean[['ราคาปกติ', 'ราคาโปร']].min(axis=1)
    allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคากลาง','ภาวะ']]

    # allcenter_clean_outstock-----------------------------------------------------------------
    allcenter_clean_outstock = allcenter_clean[allcenter_clean['ภาวะ']=='หมด'].copy()
    allcenter_clean_outstock['ราคาปกติ'] = ""
    allcenter_clean_outstock['ราคาโปร'] = ""
    allcenter_clean_outstock = allcenter_clean_outstock[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคาปกติ', 'ราคาโปร','ภาวะ']]

    allcenter_clean
    # allcenter[allcenter['ภาวะ']=='หมด']
    # output_file2 = filename[:-2] + mytext + '_checkราคากลาง.xlsx'
    allcenter_clean['Online'] = 'Online'
    allcenter_clean = allcenter_clean.sort_values(by='COMMODITY_CODE')
    # allcenter_clean.to_excel(output_file2)
    allcenter_clean.to_excel(current_dir.parent / "01-database" / "ราคากลาง" / f"Concat_ราคากลาง_{mytext}_ชุดรายเดือน.xlsx", index=False)
    print('มีข้อมูล')
except:
    columns = ['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'ราคากลาง', 'ภาวะ', 'Online']
    allcenter_clean = pd.DataFrame(columns=columns)
    allcenter_clean.to_excel(current_dir.parent / "01-database" / "ราคากลาง" / f"Concat_ราคากลาง_{mytext}_ชุดรายเดือน.xlsx", index=False)
    print('Columns เปล่า') 

มีข้อมูล


In [7]:
# allcenter_clean.columns
# import pandas as pd

# # กำหนดรายชื่อคอลัมน์
# columns = ['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'ราคากลาง', 'ภาวะ', 'Online']

# # สร้าง DataFrame เปล่า
# allcenter_cleandf = pd.DataFrame(columns=columns)
# allcenter_cleandf 